In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 00b — Clean Extraction
# MAGIC
# MAGIC **Purpose:** Extract one furnace's data from Databricks, apply all cleaning cuts,
# MAGIC and save a clean parquet file that all downstream notebooks read from.
# MAGIC
# MAGIC ## What this notebook does
# MAGIC
# MAGIC ```
# MAGIC Raw Databricks data
# MAGIC   → detect cracking runs (via feed drop)
# MAGIC   → remove: before first complete cycle
# MAGIC   → remove: decoking periods (feed ≈ 0)
# MAGIC   → remove: warm-up at run start (feed stepping up, COT unstable)
# MAGIC   → remove: pre-decoke tail at run end (feed winding down)
# MAGIC   → remove: after last complete cycle
# MAGIC   → save output/{FURNACE}_clean.parquet
# MAGIC ```
# MAGIC
# MAGIC **Output:** `output/{FURNACE}_clean.parquet` — indexed by timestamp,
# MAGIC column `run_id` labels each cracking run, column `phase` = `'clean'` everywhere.

## 1. Parameters

Change `FURNACE` to run on a different furnace. All other settings are shared.

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

# ── MUST be set before any olefins_ddf imports ────────────────────────────
os.environ['DDF_CLUSTER_ID'] = '0612-154044-62gqpkte'

from olefins_ddf.io_events import get_spark
from olefins_ddf import catalog as cat, features as feat_mod
from olefins_ddf.runs import segment_runs, cracking_mask

FURNACE       = '1HA'
START         = '2025-02-05'
END           = '2026-03-20'
BUCKET_MIN    = 30
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean')

FEED_SETTLED_FRAC = 0.95
COT_STD_THRESH_C  = 2.0
COT_ROLL_STEPS    = 8
MIN_WARMUP_H      = 12.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Furnace : {FURNACE}')
print(f'Window  : {START} → {END}')
print(f'Cluster : {os.environ["DDF_CLUSTER_ID"]}')

## 2. Connect to Databricks & load catalog

> **If cluster is sleeping:** run `databricks clusters start <id>` first.
> Update `DDF_CLUSTER_ID` env var when cluster ID changes.

In [ ]:
spark   = get_spark(os.environ['DDF_CLUSTER_ID'])
catalog = cat.build_catalog(spark)

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {DELTA_CATALOG}')
print(f'Schema ready: {DELTA_CATALOG}')

tube_tags = catalog[catalog['role'] == 'tube_COT']
if 'furnace' in tube_tags.columns:
    tube_tags = tube_tags[tube_tags['furnace'] == FURNACE]
print(f'Catalog loaded: {len(catalog)} total tags')
print(f'tube_COT for {FURNACE}: {len(tube_tags)} tags (expect 192)')

## 3. Load feature matrix

Load from **Delta table** if it already exists. Otherwise build from raw historian data (~5–10 min) and save back to Delta.
**No parquet is written to local disk.** All processed data stays in Databricks.

In [ ]:
DELTA_TABLE_FEAT = f'{DELTA_CATALOG}.{FURNACE.lower()}_features'

try:
    print(f'Loading cached feature matrix from Delta: {DELTA_TABLE_FEAT}')
    feat_spark = spark.table(DELTA_TABLE_FEAT)
    feat = feat_spark.toPandas()
    if 'timestamp' in feat.columns:
        feat = feat.set_index('timestamp')
        feat.index = pd.to_datetime(feat.index)
        feat = feat.sort_index()
    print(f'Loaded {len(feat):,} rows from Delta cache.')
except Exception as e:
    print(f'Delta cache not found ({e}). Building from Databricks (~5-10 min)...')
    feat, _ = feat_mod.build_feature_matrix(
        spark, catalog, FURNACE, start=START, end=END, bucket=BUCKET_MIN
    )
    sdf = spark.createDataFrame(feat.reset_index().rename(columns={'index': 'timestamp'}))
    (sdf.write
       .format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .saveAsTable(DELTA_TABLE_FEAT))
    print(f'Saved to Delta: {DELTA_TABLE_FEAT}')

print(f'Feature matrix : {feat.shape[0]:,} rows x {feat.shape[1]} columns')
print(f'Time range     : {feat.index.min()} -> {feat.index.max()}')
print(f'Columns        : {list(feat.columns)}')

## 4. Detect cracking runs

A run = contiguous cracking period between two decokes.
Decoke detected from **HC feed drop to ~0** (not temperature — decoke does NOT cool the furnace).

In [ ]:
# Ensure DatetimeIndex — Delta cache may store timestamp as 'ts'
if not isinstance(feat.index, pd.DatetimeIndex):
    _ts_col = next((c for c in ['timestamp', 'ts'] if c in feat.columns), None)
    if _ts_col:
        feat = feat.set_index(_ts_col)
        feat.index = pd.to_datetime(feat.index)
        feat = feat.sort_index()

feed_col = 'feed_total' if 'feed_total' in feat.columns else None
if feed_col is None:
    raise ValueError('Cannot detect runs without feed data')

mask     = cracking_mask(feat[feed_col])
runs_all = segment_runs(mask)

DATA_START = pd.Timestamp(START)
DATA_END   = pd.Timestamp(END)
MIN_GAP_H  = 24

# FIX: compare pd.Timedelta directly — avoids numpy.timedelta64 .total_seconds() issue
runs = [r for r in runs_all
        if (r.start - DATA_START) >= pd.Timedelta(hours=MIN_GAP_H)
        and (DATA_END - r.end)   >= pd.Timedelta(hours=MIN_GAP_H)]

print(f'All runs detected       : {len(runs_all)}')
print(f'Complete cycles (kept)  : {len(runs)}  (both sides bounded by decoke)')
print(f'Incomplete cycles (cut) : {len(runs_all) - len(runs)}')
print()
for r in runs:
    print(f'  Run {r.index:2d}: {r.start.date()} -> {r.end.date()}  ({r.length_days:.1f} d)')

## 5. Find analysis window for each run

For each run, find the stable cracking window:
- **Start:** feed > 95% median AND rolling std(COT) < 2 C/4h AND t >= run.start + 12h
- **End:** last timestamp where feed > 95% median

In [ ]:
def find_analysis_window(run, feat_df):
    seg  = feat_df.loc[run.start:run.end].copy()
    null = dict(run=run.index, run_start=run.start, run_end=run.end,
                analysis_start=pd.NaT, analysis_end=pd.NaT,
                warmup_hours=np.nan, tail_hours=np.nan,
                length_days=round(run.length_days, 2),
                clean_days=np.nan, settled=False)
    if seg.empty:
        return null

    earliest_start = run.start + pd.Timedelta(hours=MIN_WARMUP_H)

    run_med_feed = seg['feed_total'][seg['feed_total'] > seg['feed_total'].max() * 0.5].median()
    feed_ok = seg['feed_total'] > FEED_SETTLED_FRAC * run_med_feed

    cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in seg.columns), None)
    if cot_col:
        cot_std = seg[cot_col].rolling(COT_ROLL_STEPS, min_periods=2).std()
        cot_ok  = cot_std < COT_STD_THRESH_C
    else:
        cot_ok = pd.Series(True, index=seg.index)

    time_ok      = seg.index >= earliest_start
    settled_mask = feed_ok & cot_ok & time_ok
    settled_idx  = settled_mask[settled_mask].index

    if settled_idx.empty:
        analysis_start = earliest_start
        settled        = False
    else:
        analysis_start = settled_idx[0]
        settled        = True

    warmup_hours  = (analysis_start - run.start).total_seconds() / 3600
    end_valid_idx = feed_ok[feed_ok].index
    analysis_end  = end_valid_idx[-1] if not end_valid_idx.empty else run.end
    if analysis_end <= analysis_start:
        analysis_end = run.end

    tail_hours = (run.end - analysis_end).total_seconds() / 3600
    clean_days = (analysis_end - analysis_start).total_seconds() / 86400

    return dict(
        run=run.index, run_start=run.start, run_end=run.end,
        analysis_start=analysis_start, analysis_end=analysis_end,
        warmup_hours=round(warmup_hours, 1), tail_hours=round(tail_hours, 1),
        length_days=round(run.length_days, 2), clean_days=round(clean_days, 2),
        settled=settled,
    )


windows = [find_analysis_window(r, feat) for r in runs]
win_df  = pd.DataFrame(windows)

print(f'Windows computed for {len(win_df)} runs')
print(f'  Settled    : {win_df["settled"].sum()} / {len(win_df)}')
print(f'  Warmup avg : {win_df["warmup_hours"].mean():.1f} h')
print(f'  Tail avg   : {win_df["tail_hours"].mean():.1f} h')
print(f'  Clean avg  : {win_df["clean_days"].mean():.1f} d')
display(win_df[['run','run_start','run_end','analysis_start','analysis_end',
                'warmup_hours','tail_hours','clean_days','settled']])

## 6. Apply all cuts — build clean time series

Four things removed:
1. **Before first complete cycle**
2. **Decoking periods** — feed ~ 0
3. **Warm-up** — feed stepping up, COT unstable
4. **Pre-decoke tail** — feed winding down

In [ ]:
clean_mask = pd.Series(False, index=feat.index)
run_id_col = pd.Series(np.nan,  index=feat.index)

valid_runs = win_df[win_df['analysis_start'].notna() & win_df['analysis_end'].notna()].copy()

for _, w in valid_runs.iterrows():
    a_start = w['analysis_start']
    a_end   = w['analysis_end']
    if pd.isna(a_start) or pd.isna(a_end) or a_end <= a_start:
        continue
    in_window = (feat.index >= a_start) & (feat.index <= a_end)
    clean_mask[in_window] = True
    run_id_col[in_window] = int(w['run'])

feat_clean = feat[clean_mask].copy()
feat_clean['run_id'] = run_id_col[clean_mask].astype(int)
feat_clean['phase']  = 'clean'

total_rows = len(feat)
clean_rows = len(feat_clean)
pct_kept   = 100 * clean_rows / total_rows

print(f'Total rows : {total_rows:,}')
print(f'Kept       : {clean_rows:,}  ({pct_kept:.1f}%)')
print(f'Removed    : {total_rows - clean_rows:,}  ({100-pct_kept:.1f}%)')
print(f'Runs       : {feat_clean["run_id"].nunique()}')

## 7. Manual exclusions

Add known sensor faults that survived the automated filter.
Harry's principle: only remove physically impossible events (not real high-DD periods).

> **1HA known issue:** spike ~120 C in dd_abs_max (Oct/Nov 2025, Run 9).
> Needs DCS verification before excluding — leave commented until confirmed.

In [ ]:
MANUAL_EXCLUDE = [
    # ('Run9_sensor_fault', '2025-10-28', '2025-11-02'),
]

excluded_count = 0
for reason, t0, t1 in MANUAL_EXCLUDE:
    mask_exc = (feat_clean.index >= t0) & (feat_clean.index <= t1)
    n = mask_exc.sum()
    if 'dd_abs_max' in feat_clean.columns:
        feat_clean.loc[mask_exc, 'dd_abs_max'] = np.nan
    excluded_count += n
    print(f'  Excluded: {n} rows — {reason}')

if not MANUAL_EXCLUDE:
    print('No manual exclusions applied.')

## 8. Save to Databricks Delta

Clean data written as Delta table — stays in Databricks, never on local disk.
Downstream notebooks read it via `spark.table()`.

In [ ]:
DELTA_TABLE_CLEAN = f'{DELTA_CATALOG}.{FURNACE.lower()}_clean'

sdf_clean = spark.createDataFrame(
    feat_clean.reset_index().rename(columns={'index': 'timestamp'})
)
(sdf_clean.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DELTA_TABLE_CLEAN))

print(f'Clean data saved : {DELTA_TABLE_CLEAN}')
print(f'Shape            : {feat_clean.shape}')

win_path = os.path.join(OUTPUT_DIR, 'run_analysis_windows.csv')
win_df.to_csv(win_path, index=False)
print(f'Run windows CSV  : {win_path}')

## 9. Verification plot

Visual check: green = kept, red = removed.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'{FURNACE} — Clean extraction summary (green=kept, red=removed)',
             fontsize=12, fontweight='bold')

ax1, ax2, ax3 = axes

if 'feed_total' in feat.columns:
    ax1.plot(feat.index,       feat['feed_total'],       color='#378add', lw=0.6, alpha=0.7, label='raw')
    ax1.plot(feat_clean.index, feat_clean['feed_total'], color='#1d9e75', lw=0.8,            label='clean')
ax1.set_ylabel('Feed total (NM3/H)')
ax1.legend(fontsize=8, loc='upper right')

cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in feat.columns), None)
if cot_col:
    ax2.plot(feat.index,       feat[cot_col],       color='#7f77dd', lw=0.6, alpha=0.7, label='raw')
    ax2.plot(feat_clean.index, feat_clean[cot_col], color='#085041', lw=0.8,            label='clean')
ax2.set_ylabel('COT (C)')
ax2.legend(fontsize=8, loc='upper right')

if 'dd_abs_max' in feat.columns:
    ax3.plot(feat.index,       feat['dd_abs_max'],       color='#ef9f27', lw=0.6, alpha=0.5, label='raw')
    ax3.plot(feat_clean.index, feat_clean['dd_abs_max'], color='#d85a30', lw=0.9,            label='clean')
    ax3.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45C')
    ax3.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30C')
ax3.set_ylabel('dd_abs_max (C)')
ax3.legend(fontsize=8, loc='upper right')

for _, w in valid_runs.iterrows():
    for ax in axes:
        ax.axvspan(w['run_start'],      w['analysis_start'], alpha=0.12, color='#e24b4a', zorder=0)
        ax.axvspan(w['analysis_start'], w['analysis_end'],   alpha=0.10, color='#1d9e75', zorder=0)
        ax.axvspan(w['analysis_end'],   w['run_end'],        alpha=0.12, color='#e24b4a', zorder=0)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, f'{FURNACE}_00b_clean_extraction.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

## 10. Summary

In [ ]:
print('=' * 60)
print(f'CLEAN EXTRACTION SUMMARY — {FURNACE}')
print('=' * 60)
print(f'  Furnace          : {FURNACE}')
print(f'  Raw data window  : {START} -> {END}')
print(f'  Total runs found : {len(runs)}')
print(f'  Complete cycles  : {len(valid_runs)}')
print(f'  Clean rows kept  : {clean_rows:,}  ({pct_kept:.1f}% of raw)')
print(f'  Total clean time : {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')
print()
print(f'  Feature matrix : {DELTA_TABLE_FEAT}')
print(f'  Clean data     : {DELTA_TABLE_CLEAN}')
print(f'  Run windows    : {win_path}')
print()
print('How to use in downstream notebooks:')
print(f'  feat_clean = spark.table("{DELTA_TABLE_CLEAN}").toPandas()')
print( '  feat_clean = feat_clean.set_index("timestamp")')
print()

suspicious = win_df[(win_df['warmup_hours'] > 48) | (win_df['tail_hours'] > 48)]
if len(suspicious):
    print('Warning — suspicious runs (verify with DCS):')
    for _, s in suspicious.iterrows():
        print(f'  Run {int(s["run"]):2d}: warmup={s["warmup_hours"]:.0f}h, tail={s["tail_hours"]:.0f}h')

## 11. Reload from Delta (standalone)

Run this cell alone to reload all outputs from Delta without re-running the full pipeline.
Useful when you only want to regenerate plots.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT  = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)
os.environ['DDF_CLUSTER_ID'] = '0612-154044-62gqpkte'

from olefins_ddf.io_events import get_spark

FURNACE       = '1HA'
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'
OUTPUT_DIR    = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean')

spark = get_spark(os.environ['DDF_CLUSTER_ID'])

feat = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_features').toPandas()
ts_col = next(c for c in ['timestamp', 'ts'] if c in feat.columns)
feat = feat.set_index(ts_col)
feat.index = pd.to_datetime(feat.index)
feat = feat.sort_index()
feat.index.name = 'timestamp'

feat_clean = spark.table(f'{DELTA_CATALOG}.{FURNACE.lower()}_clean').toPandas()
ts_col = next(c for c in ['timestamp', 'ts'] if c in feat_clean.columns)
feat_clean = feat_clean.set_index(ts_col)
feat_clean.index = pd.to_datetime(feat_clean.index)
feat_clean = feat_clean.sort_index()
feat_clean.index.name = 'timestamp'

win_path   = os.path.join(OUTPUT_DIR, 'run_analysis_windows.csv')
valid_runs = pd.read_csv(win_path, parse_dates=['run_start', 'run_end', 'analysis_start', 'analysis_end'])

print(f'feat       : {feat.shape}')
print(f'feat_clean : {feat_clean.shape}')
print(f'runs       : {len(valid_runs)}')

## 12. Per-cycle plots (run-age aligned)

One plot per cracking run. X-axis = days since last decoke (same scale across all runs).
Panels: Feed, COT, dd_abs_max (furnace), dd_abs_max per pass (A/B/C/D).
Grey zone = warm-up, orange zone = pre-decoke tail.
Title in red = run needs DCS verification.

In [ ]:
import os, matplotlib.pyplot as plt

CYCLE_DIR   = os.path.join(OUTPUT_DIR, 'cycles')
PASS_COLORS = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}

os.makedirs(CYCLE_DIR, exist_ok=True)

X_MAX   = valid_runs['length_days'].max() + 1.0
cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in feat_clean.columns), None)

for _, w in valid_runs.iterrows():
    rid = int(w['run'])
    seg = feat_clean[feat_clean['run_id'] == rid].copy()

    if seg.empty or 'run_age_days' not in seg.columns:
        print(f'Run {rid}: skipped (no data or run_age_days missing)')
        continue

    x        = seg['run_age_days']
    warmup_d = w['warmup_hours'] / 24
    tail_d   = w['tail_hours']   / 24
    end_d    = w['length_days']
    flag     = ' WARNING VERIFY DCS' if (w['warmup_hours'] > 48 or w['tail_hours'] > 48) else ''

    pass_cols = {p: f'dd_abs_max_{p}' for p in 'ABCD' if f'dd_abs_max_{p}' in seg.columns}
    n_panels  = 4 if pass_cols else 3

    fig, axes = plt.subplots(n_panels, 1, figsize=(14, 3.2 * n_panels), sharex=True)
    fig.suptitle(
        f'{FURNACE}  Run {rid}  |  {w["run_start"].date()} -> {w["run_end"].date()}'
        f'  ({end_d:.1f}d total, {w["clean_days"]:.1f}d clean'
        f', warmup {w["warmup_hours"]:.0f}h, tail {w["tail_hours"]:.0f}h){flag}',
        fontsize=10, fontweight='bold',
        color='#c0392b' if flag else 'black',
    )

    ax1, ax2, ax3 = axes[0], axes[1], axes[2]

    if 'feed_total' in seg.columns:
        ax1.plot(x, seg['feed_total'], color='teal', lw=0.8)
    ax1.set_ylabel('Feed (NM3/H)')

    if cot_col:
        ax2.plot(x, seg[cot_col], color='darkgreen', lw=0.8)
    ax2.set_ylabel('COT (C)')

    if 'dd_abs_max' in seg.columns:
        ax3.plot(x, seg['dd_abs_max'], color='tomato', lw=0.9)
    ax3.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45C')
    ax3.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30C')
    ax3.set_ylabel('dd_abs_max (C)')
    ax3.legend(fontsize=7, loc='upper right')

    if n_panels == 4:
        ax4 = axes[3]
        for p, col_name in pass_cols.items():
            ax4.plot(x, seg[col_name], color=PASS_COLORS[p], lw=0.7, label=f'Pass {p}')
        ax4.axhline(45, color='#e24b4a', ls='--', lw=1.0)
        ax4.axhline(30, color='#ef9f27', ls='--', lw=1.0)
        ax4.set_ylabel('dd per pass (C)')
        ax4.legend(fontsize=7, loc='upper right', ncol=2)

    for ax in axes:
        ax.set_xlim(0, X_MAX)
        ax.axvspan(0, warmup_d, alpha=0.10, color='grey')
        ax.axvline(warmup_d,    color='grey',   lw=0.8, ls='--')
        if tail_d > 0:
            ax.axvspan(end_d - tail_d, end_d, alpha=0.10, color='orange')
            ax.axvline(end_d - tail_d, color='orange', lw=0.8, ls='--')

    axes[-1].set_xlabel('Run age (days)')
    plt.tight_layout()
    save_path = os.path.join(CYCLE_DIR, f'{FURNACE}_run{rid:02d}.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Run {rid:2d} -> {save_path}')

print(f'\nDone — {len(valid_runs)} cycle plots saved to {CYCLE_DIR}')